[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C25_Long_Context_Course/04_linear_ssm/04_linear_ssm.ipynb)

# 04 · 线性注意力与 SSM（用 numpy 从零写出）

**算力墙的另一条路**：稀疏(模块03)仍保留 softmax 形式；线性注意力**彻底换掉 softmax**，用核映射 φ + 结合律把 O(n²) 降成 **O(n)**，
因果版等价于一个固定大小状态的 **RNN**。代价：φ 只近似 softmax、固定状态有损压缩历史。

**路线**：
1. softmax vs 去 softmax：为什么 softmax 逼着我们算 n×n
2. **线性注意力（非因果）**：φ(Q)(φ(K)ᵀV) 结合律 → 对拍二次形式
3. 复杂度对比：n×n vs d×d
4. **因果递推**：状态 S 累加 = RNN → 对拍因果二次形式
5. **SSM 扫描**：h_t=A·h_{t-1}+B·x_t 的线性递推
6. 线性的软肋：精确检索上不如 softmax
7. ✏️ 练习（φ / 状态累加 / causal 线性 / 复杂度计数）→ 📖 答案 → 🧪 胶囊

> 本课纪律：线性注意力的 O(n) 实现必须**对拍它自己的二次形式**（结合律是恒等变换，结果应一致）。

## 1 · 为什么 softmax 逼着我们算 n×n

softmax 注意力 = `softmax(QKᵀ)V`。softmax 是逐行非线性，必须先有完整的 `QKᵀ`(n×n) 才能做。
**去掉 softmax** 后是 `(QKᵀ)V`，矩阵乘有结合律 → 可改成 `Q(KᵀV)`，先算 d×d 的 KᵀV，绕开 n×n！

In [ ]:
import numpy as np
rng = np.random.default_rng(0)

n, d = 8, 4
Q = rng.standard_normal((n, d)); K = rng.standard_normal((n, d)); V = rng.standard_normal((n, d))

# 去 softmax 的注意力：两种算法应数值相等（结合律）
left  = (Q @ K.T) @ V          # 先 QK^T (n×n) 再 ·V   —— O(n^2 d)
right = Q @ (K.T @ V)          # 先 K^T V (d×d) 再 Q·  —— O(n d^2)
assert np.allclose(left, right, atol=1e-10), '结合律：两种顺序结果相同'
print('(QK^T)V 形状', left.shape, ' == Q(K^T V) 形状', right.shape)
print('✅ 没有 softmax 时，结合律让我们先算 d×d 的 K^T V，绕开 n×n —— 这是线性化的全部秘密')
print('   但 softmax 是非线性，不能直接挪到结合律外 → 需要用 φ 近似（下一节）')

## 2 · 线性注意力（非因果）：φ + 结合律

用特征映射 `φ`(这里 elu+1，保证非负) 替代 softmax 的指数核：`exp(qᵀk) ≈ φ(q)ᵀφ(k)`。
则 `o_i = φ(q_i)ᵀS / (φ(q_i)ᵀz)`，其中 `S=Σφ(k_j)v_jᵀ`(d×d)、`z=Σφ(k_j)`(d)。
**对拍**：O(n) 的结合律版 == O(n²) 的二次版（同一个 φ，结合律是恒等变换）。

In [ ]:
def feature_map(x):
    '''φ(x)=elu(x)+1，恒正（替代 exp 的恒正相似度）。'''
    return np.where(x > 0, x + 1.0, np.exp(x))     # elu(x)+1

def linear_attn_quadratic(Q, K, V):
    '''二次形式：显式算 φ(Q)φ(K)ᵀ (n×n) 再归一化乘 V。O(n²)。'''
    Qp, Kp = feature_map(Q), feature_map(K)
    A = Qp @ Kp.T                                  # (n,n) 非负相似度
    A = A / A.sum(axis=1, keepdims=True)           # 行归一化（替代 softmax）
    return A @ V

def linear_attn_linear(Q, K, V):
    '''结合律版：先算 S=φ(K)ᵀV (d×d) 和 z=Σφ(K)。O(n)。'''
    Qp, Kp = feature_map(Q), feature_map(K)
    S = Kp.T @ V                                   # (d,d)  = Σ φ(k_j) v_jᵀ
    z = Kp.sum(axis=0)                             # (d,)   = Σ φ(k_j)
    num = Qp @ S                                   # (n,d)
    den = Qp @ z                                   # (n,)
    return num / den[:, None]

n, d = 10, 4
Q = rng.standard_normal((n,d)); K = rng.standard_normal((n,d)); V = rng.standard_normal((n,d))
o_quad = linear_attn_quadratic(Q, K, V)
o_lin  = linear_attn_linear(Q, K, V)
print('二次形式 vs 结合律形式 最大误差:', np.abs(o_quad - o_lin).max())
assert np.allclose(o_quad, o_lin, atol=1e-10), '结合律是恒等变换，两者应相等'
print('✅ O(n) 的结合律版与 O(n²) 的二次版逐位相同 —— n×n 矩阵被绕开了')

## 3 · 复杂度对比：n×n vs d×d

二次版要造 n×n（随 n 平方）；线性版只造 d×d（与 n 无关）。数一下两者的「大矩阵元素数」随 n 的变化。

In [ ]:
def quad_big_matrix(n, d):  return n * n          # φ(Q)φ(K)ᵀ
def lin_big_matrix(n, d):   return d * d          # φ(K)ᵀV（与 n 无关！）

d = 64
print(f"{'n':>8s} {'二次 n×n':>14s} {'线性 d×d':>12s} {'倍数':>12s}")
for n in [1024, 16384, 262144]:
    q, l = quad_big_matrix(n, d), lin_big_matrix(n, d)
    print(f'{n:>8d} {q:>14d} {l:>12d} {q/l:>11.0f}x')
assert lin_big_matrix(262144, d) == lin_big_matrix(1024, d), '线性版大矩阵与 n 无关'
assert quad_big_matrix(2048, d) == 4 * quad_big_matrix(1024, d), '二次版随 n 平方'
print('\n✅ 线性版的核心矩阵 d×d 完全不随 n 增长 —— 这是 O(n) 的来源')

## 4 · 因果递推：状态 S 累加 = RNN

因果下 query i 只看 j≤i：用**前缀和** `S_i=Σ_{j≤i}φ(k_j)v_jᵀ`，递推维护 `S_i=S_{i-1}+φ(k_i)v_iᵀ`。
这是一个固定大小状态(d×d)的 **RNN**！对拍因果二次形式。

In [ ]:
def causal_linear_quadratic(Q, K, V):
    '''因果二次：φ(Q)φ(K)ᵀ 加下三角掩码再行归一化。'''
    n = Q.shape[0]
    Qp, Kp = feature_map(Q), feature_map(K)
    A = Qp @ Kp.T
    A = np.tril(A)                                 # 因果：只保留 j<=i
    A = A / A.sum(axis=1, keepdims=True)
    return A @ V

def causal_linear_recurrent(Q, K, V):
    '''因果递推：逐步更新状态 S(d×d) 和 z(d)，每步 O(d²)。总 O(n d²)。'''
    n, d = Q.shape
    Qp, Kp = feature_map(Q), feature_map(K)
    S = np.zeros((d, d)); z = np.zeros(d)
    out = np.zeros((n, d))
    for i in range(n):
        S = S + np.outer(Kp[i], V[i])             # 状态更新 S_i = S_{i-1} + φ(k_i)v_iᵀ
        z = z + Kp[i]
        out[i] = (Qp[i] @ S) / (Qp[i] @ z)        # 读出 o_i
    return out

n, d = 12, 4
Q = rng.standard_normal((n,d)); K = rng.standard_normal((n,d)); V = rng.standard_normal((n,d))
o_quad = causal_linear_quadratic(Q, K, V)
o_rec  = causal_linear_recurrent(Q, K, V)
print('因果二次 vs 因果递推 最大误差:', np.abs(o_quad - o_rec).max())
assert np.allclose(o_quad, o_rec, atol=1e-10), '递推(RNN)应等于因果二次形式'
print('✅ 因果线性注意力 = 固定状态 RNN：每步 O(d²)、状态大小与序列长无关')
print('   解码时无需 KV cache（换成一个 d×d 状态）—— 对长序列推理是根本优势')

## 5 · SSM 扫描：h_t = A·h_{t-1} + B·x_t

「固定状态 + 线性递推」更一般的形式就是 SSM。实现一个对角 SSM（A 是衰减因子），
它和线性注意力同构——都是流式维护固定状态。验证：A=1(不衰减) 时退化为前缀和（累积）。

In [ ]:
def ssm_scan(x, A, B, C):
    '''对角 SSM: h_t = A*h_{t-1} + B*x_t, y_t = C*h_t。x:(n,), A/B/C 标量或(n,)。'''
    n = len(x)
    h = 0.0; y = np.zeros(n)
    A = np.broadcast_to(A, (n,)); B = np.broadcast_to(B, (n,)); C = np.broadcast_to(C, (n,))
    for t in range(n):
        h = A[t] * h + B[t] * x[t]                # 状态递推
        y[t] = C[t] * h                           # 读出
    return y

x = rng.standard_normal(8)
# A=1, B=1, C=1 → h_t = h_{t-1} + x_t = 前缀和
y = ssm_scan(x, 1.0, 1.0, 1.0)
assert np.allclose(y, np.cumsum(x)), 'A=1 的 SSM = 前缀和'
print('A=1 (不衰减): SSM 输出 = 累积和 ✅')
# A=0.5 (衰减): 旧信息按比例遗忘 —— 模拟「近处更重要」
y_decay = ssm_scan(x, 0.5, 1.0, 1.0)
assert abs(y_decay[-1] - (0.5*y_decay[-2] + x[-1])) < 1e-12, '验证衰减递推'
print('A=0.5 (衰减): 旧信息按比例遗忘（RetNet/GLA 的门控思想）✅')
print('✅ SSM 与线性注意力同构：都是固定状态的线性扫描；A 控制记多久')

## 6 · 线性的软肋：精确检索不如 softmax

把无限历史压进固定状态必然丢信息。构造一个「精确检索」场景：让 query 想精确取出某个远处 key 的 value，
对比 softmax（能尖锐聚焦）与线性注意力（φ 近似，聚焦能力弱）。

In [ ]:
def softmax_attn(Q, K, V):
    S = Q @ K.T / np.sqrt(Q.shape[1])
    P = np.exp(S - S.max(1, keepdims=True)); P /= P.sum(1, keepdims=True)
    return P @ V

n, d = 16, 8
K = rng.standard_normal((n, d)); V = rng.standard_normal((n, d))
# query 几乎等于第 target 个 key（想精确检索它的 value）
target = 3
q = K[target] * 3.0                               # 放大 → softmax 会尖锐聚焦到 target
Q = q[None, :]
o_soft = softmax_attn(Q, K, V)[0]
o_lin  = linear_attn_linear(Q, K, V)[0]
err_soft = np.linalg.norm(o_soft - V[target])
err_lin  = np.linalg.norm(o_lin  - V[target])
print(f'检索误差(越小越准)：softmax={err_soft:.3f}  线性={err_lin:.3f}')
assert err_soft < err_lin, 'softmax 能更尖锐地聚焦 → 精确检索更准'
print('✅ softmax 能尖锐聚焦单个 key，线性注意力(φ 近似)聚焦弱 → 精确检索吃亏')
print('   这是线性模型在 NIAH/RULER(模块05) 上常输给 softmax 的根本原因')

---
## ✏️ 练习 1：实现一个合法的特征映射 φ

实现 `phi_relu(x)`：用 `relu(x)+ε`（ε 小正数保证恒正）作为特征映射。
验证：输出恒正（替代 exp 的恒正性）、保持形状。

In [ ]:
def phi_relu(x, eps=1e-3):
    # TODO: 返回 maximum(x, 0) + eps（恒正）
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
x = rng.standard_normal((5, 4))
px = phi_relu(x)
assert px.shape == x.shape
assert np.all(px > 0), 'φ 必须恒正（替代 exp 的恒正相似度）'
# 恒正保证 φ(q)ᵀφ(k) 非负 → 注意力权重非负
sim = phi_relu(rng.standard_normal(4)) @ phi_relu(rng.standard_normal(4))
assert sim > 0, 'φ(q)·φ(k) 应非负'
print('✅ 练习 1 通过：恒正特征映射（注意力权重非负的保证）')

## ✏️ 练习 2：线性注意力的状态累加（结合律版）

实现非因果 `lin_attn(Q,K,V,phi)`：用结合律先算 `S=φ(K)ᵀV`、`z=Σφ(K)`，再 `o=φ(Q)S/(φ(Q)z)`。
目标：对拍二次形式（用同一个 φ）。

In [ ]:
def lin_attn(Q, K, V, phi):
    Qp, Kp = phi(Q), phi(K)
    # TODO: S = Kp.T @ V ; z = Kp.sum(0) ; 返回 (Qp@S) / (Qp@z)[:,None]
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
def quad_ref(Q, K, V, phi):
    A = phi(Q) @ phi(K).T; A = A / A.sum(1, keepdims=True); return A @ V
n, d = 10, 4
Q = rng.standard_normal((n,d)); K = rng.standard_normal((n,d)); V = rng.standard_normal((n,d))
assert np.allclose(lin_attn(Q,K,V,feature_map), quad_ref(Q,K,V,feature_map), atol=1e-10)
print('✅ 练习 2 通过：结合律版 O(n) == 二次版 O(n²)（恒等变换）')

## ✏️ 练习 3：因果线性注意力的递推（RNN）

实现 `causal_lin_rec(Q,K,V,phi)`：逐步维护状态 `S(d×d)` 和 `z(d)`，每步累加 `φ(k_i)v_iᵀ` 并读出 `o_i`。
目标：对拍因果二次形式。这就是「线性注意力 = RNN」。

In [ ]:
def causal_lin_rec(Q, K, V, phi):
    n, d = Q.shape; Qp, Kp = phi(Q), phi(K)
    S = np.zeros((d, d)); z = np.zeros(d); out = np.zeros((n, d))
    # TODO: for i in range(n): S += outer(Kp[i],V[i]); z += Kp[i]; out[i]=(Qp[i]@S)/(Qp[i]@z)
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
def causal_quad_ref(Q, K, V, phi):
    A = np.tril(phi(Q) @ phi(K).T); A = A / A.sum(1, keepdims=True); return A @ V
n, d = 14, 4
Q = rng.standard_normal((n,d)); K = rng.standard_normal((n,d)); V = rng.standard_normal((n,d))
assert np.allclose(causal_lin_rec(Q,K,V,feature_map), causal_quad_ref(Q,K,V,feature_map), atol=1e-10)
print('✅ 练习 3 通过：因果递推(RNN, 固定状态) == 因果二次形式')

## ✏️ 练习 4：数复杂度——线性 vs 二次的 FLOPs

实现 `flops_ratio(n, d)`：返回「二次注意力 FLOPs ÷ 线性注意力 FLOPs」。
二次 ≈ `2·n²·d`（QKᵀ + ·V）；线性 ≈ `2·n·d²`（KᵀV + Q·）。验证：n≫d 时线性大胜。

In [ ]:
def flops_ratio(n, d):
    # TODO: quad = 2*n*n*d ; lin = 2*n*d*d ; 返回 quad/lin  (= n/d)
    raise NotImplementedError

In [ ]:
# —— 练习 4 自测 ——
assert abs(flops_ratio(8192, 64) - 8192/64) < 1e-6, '比值应 = n/d'
r_short = flops_ratio(512, 64)
r_long  = flops_ratio(262144, 64)
assert r_long > r_short, '序列越长，线性优势越大'
print(f'FLOPs 倍数(二次/线性)：n=512 时 {r_short:.0f}x，n=256k 时 {r_long:.0f}x')
print('✅ 练习 4 通过：比值 = n/d，长序列下线性注意力算力优势巨大')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def phi_relu(x, eps=1e-3):
    return np.maximum(x, 0.0) + eps

In [ ]:
# 练习 2 参考答案
def lin_attn(Q, K, V, phi):
    Qp, Kp = phi(Q), phi(K)
    S = Kp.T @ V
    z = Kp.sum(0)
    return (Qp @ S) / (Qp @ z)[:, None]

In [ ]:
# 练习 3 参考答案
def causal_lin_rec(Q, K, V, phi):
    n, d = Q.shape; Qp, Kp = phi(Q), phi(K)
    S = np.zeros((d, d)); z = np.zeros(d); out = np.zeros((n, d))
    for i in range(n):
        S = S + np.outer(Kp[i], V[i])
        z = z + Kp[i]
        out[i] = (Qp[i] @ S) / (Qp[i] @ z)
    return out

In [ ]:
# 练习 4 参考答案
def flops_ratio(n, d):
    quad = 2 * n * n * d
    lin  = 2 * n * d * d
    return quad / lin

---
## 🧪 真实数据胶囊：线性模型解码省了多少 KV 内存？

softmax 注意力解码时 KV cache 随长度线性膨胀；线性/SSM 是固定状态。
用真实配置算：处理到长度 n 时，softmax 的 KV cache vs 线性模型的固定状态，差多少？

In [ ]:
def kv_vs_state_gb(n, n_layers=32, n_heads=32, d=128, state_per_layer_mb=4, b=2):
    # TODO: 
    #   softmax KV cache(GB) = 2*n_layers*n_heads*d*n*b / 1e9
    #   线性固定状态(GB) = n_layers*state_per_layer_mb/1024 （与 n 无关！）
    #   返回 (kv_gb, state_gb)
    raise NotImplementedError

In [ ]:
# 自测
print(f"{'序列长 n':>10s} {'softmax KV(GB)':>16s} {'线性状态(GB)':>14s}")
for n in [8192, 131072, 1_000_000]:
    kv, st = kv_vs_state_gb(n)
    print(f'{n:>10d} {kv:>16.2f} {st:>14.3f}')
kv_1m, st_1m = kv_vs_state_gb(1_000_000)
kv_8k, st_8k = kv_vs_state_gb(8192)
assert kv_1m > kv_8k, 'softmax KV 随 n 增长'
assert abs(st_1m - st_8k) < 1e-9, '线性状态与 n 无关（固定）'
assert st_1m < kv_1m, '1M 时线性状态远小于 softmax KV'
print('\n✅ softmax KV cache 随 n 线性膨胀(1M 时数百GB)，线性模型状态固定不变 → 长解码的根本优势')
print('   代价：固定状态有损压缩历史，精确检索(NIAH/RULER)上常不如 softmax —— 见模块 05。')

In [ ]:
# 📖 胶囊参考答案
def kv_vs_state_gb(n, n_layers=32, n_heads=32, d=128, state_per_layer_mb=4, b=2):
    kv_gb = 2 * n_layers * n_heads * d * n * b / 1e9
    state_gb = n_layers * state_per_layer_mb / 1024
    return kv_gb, state_gb

### 小结
- softmax 是非线性，逼着我们 materialize n×n。**去 softmax + 核映射 φ** → 用结合律先算 d×d 的 φ(K)ᵀV，绕开 n×n。
- 复杂度 **O(n²d) → O(nd²)**；核心矩阵 d×d 与序列长无关。
- **因果版 = 固定状态的 RNN**：状态 S(d×d) 递推累加，解码每步 O(d²)、**无 KV cache**（换成固定状态）。
- **SSM/Mamba**：更一般的「固定状态 + 线性扫描」；Mamba 的选择性(参数随输入变)补上内容寻址软肋。
- **根本权衡**：固定状态有损压缩历史 → 精确检索(NIAH/RULER)上常不如 softmax。前沿是**混合架构**(多数层线性 + 少数层 softmax)。

下一站：**模块 05 · KV 压缩与长上下文评测** —— 若坚持用 softmax，解码时的 KV cache 怎么压？压完还看得见吗？